In [0]:
#%pip install openpyxl

In [0]:
import pandas as pd
import plotly.express as px
routes_raw = pd.read_csv("../streamlit_app/routes.csv")
routes_raw.head()

Explore dataset

In [0]:
routes_raw.describe()

In [0]:
routes_raw.info()

Prepare data

In [0]:
# Add carbon emissions to a new column each, convert to kg
routes = routes_raw.sort_values(by='distance_km', ascending=True)
routes['air_co2_kg'] = routes['distance_km']*171/1000 # value from 'World in Data' - 171g co2/km air travel
routes['rail_co2_kg'] = routes['distance_km']*35/1000 # value from 'World in Data' - 35g co2/km rail travel
routes['rail_co2_kg_efficient'] = routes['distance_km']*5/1000 # value from 'World in Data' - 5g co2/km rail travel

# Convert time variables to hours
routes['air_time_hrs'] = routes['air_time_mins']/60
routes['rail_time_hrs'] = routes['rail_time_mins']/60
routes.head()

# Clean up cheap_flight variable
routes['cheap_flight'] = routes['cheap_flight'].fillna(0)

# Drop distance_train column
routes = routes.drop(columns=['distance_train'])

routes.head()

Understand dataset structure

In [0]:
# Histogram of distance_km to see distribution of distances

# Define reasonable distance categories (bins)
distance_bins = [0, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000, routes['distance_km'].max() + 1]
distance_labels = ['0-200', '200-400', '400-600', '600-800', '800-1000', '1000-1200', '1200-1400', '1400-1600', '1600-1800', '1800-2000', '2000+']

routes['distance_category'] = pd.cut(routes['distance_km'], bins=distance_bins, labels=distance_labels, right=False)

fig = px.histogram(routes, x='distance_category', category_orders={'distance_category': distance_labels}, 
                   title='Histogram of Route Distances', labels={'distance_category': 'Distance Category (km)'})
fig.show()

In [0]:
# Histogram of distance_km to see distribution of distances

# Define reasonable distance categories (bins)
distance_bins = [0, 250, 500, 750, 1000, 1250, 1500, 1750, 2000, routes['distance_km'].max() + 1]
distance_labels = ['0-250', '250-500', '500-750', '750-1000', '1000-1250', '1250-1500', '1500-1750', '1750-2000', '2000+']

routes['distance_category'] = pd.cut(routes['distance_km'], bins=distance_bins, labels=distance_labels, right=False)

fig = px.histogram(routes, x='distance_category', category_orders={'distance_category': distance_labels}, 
                   title='Histogram of Route Distances', labels={'distance_category': 'Distance Category (km)'})
fig.show()

In [0]:
# Divide routes into distance categories: 1-250km is short distance, 250-500km is medium distance, 500-750km is long distance, 750+ km is ultra long distance
routes['distance_label'] = pd.cut(
    routes['distance_km'],
    bins=[0, 250, 500, 750, routes['distance_km'].max() + 1],
    labels=[1, 2, 3, 4],
    right=False
).astype(int)
routes.tail()


Exploration of dataset

In [0]:
time_gap = routes['rail_time_hrs'] - routes['air_time_hrs']
price_gap = routes['rail_fare_eur'] - routes['air_fare_eur']
routes['time_gap'] = time_gap
routes['price_gap'] = price_gap

fig = px.scatter(
    routes, x=time_gap, y=price_gap,
    hover_data=['route'],
    title='Time Gap vs Price Gap - Does slower also mean cheaper?',
    labels={'x': 'Rail Time - Air Time (hrs)', 'y': 'Rail Fare - Air Fare (€)'}
)
fig.show()

# anything above the 0 line (y) means rail is more expensive. Anything below the 0 line means rail is cheaper. Virtually all trains take longer than the flight.

In [0]:
# Includes "normal" price gap

def calculate_time_gap_adjusted(air_overhead=3.0, rail_overhead=0.5):
    time_gap_adjusted = routes['rail_time_hrs'] - routes['air_time_hrs'] - air_overhead + rail_overhead
    return time_gap_adjusted

fig_adjusted = px.scatter(
    routes, x=calculate_time_gap_adjusted(air_overhead=4.0, rail_overhead=0.5), y=price_gap,
    hover_data=['route'],
    title='Slower does not always mean cheaper. Actually, rail is often faster',
    labels={'x': 'Time Gap: Rail Time - Air Time (hrs)', 'y': 'Price Gap: Rail Fare - Air Fare (€)'}
)

fig_adjusted.add_vline(x=0, line_dash="dash", line_color="black")
fig_adjusted.add_hline(y=0, line_dash="dash", line_color="black")

# Annotations for each quadrant
x_vals = calculate_time_gap_adjusted(air_overhead=3.0, rail_overhead=0.5)
y_vals = price_gap

x_min, x_max = x_vals.min(), x_vals.max()
y_min, y_max = y_vals.min(), y_vals.max()

# Midpoints of each quadrant
x_pos_right = (0 + x_max) / 2
x_pos_left  = (0 + x_min-2) / 2
y_pos_top   = (0 + y_max+1) / 2
y_pos_bottom= (0 + y_min-1) / 2

# Add annotations in correct quadrants
fig_adjusted.add_annotation(x=x_pos_right, y=y_pos_top,
                            text="Slower & <br>More Expensive", showarrow=False)
fig_adjusted.add_annotation(x=x_pos_left, y=y_pos_top,
                            text="Faster & <br>More Expensive", showarrow=False)
fig_adjusted.add_annotation(x=x_pos_left, y=y_pos_bottom,
                            text="Faster & <br>Cheaper", showarrow=False)
fig_adjusted.add_annotation(x=x_pos_right, y=y_pos_bottom,
                            text="Slower & <br>Cheaper", showarrow=False)

fig_adjusted.show()

In [0]:
def calculate_price_gap_adjusted(baggage_fee=23):
    price_gap_adjusted = routes['rail_fare_eur'] - routes['air_fare_eur'] - (routes['cheap_flight'])*baggage_fee
    return price_gap_adjusted

In [0]:
# Does something change when we use price_gap_adjusted as well?

# Calculate number of routes in each quadrant from the adjusted scatter plot, display the routes
x_vals = calculate_time_gap_adjusted(air_overhead=3.0, rail_overhead=0.5)
y_vals = calculate_price_gap_adjusted(baggage_fee=23)

quadrant_masks = {
    "Slower & More Expensive": (x_vals > 0) & (y_vals > 0),
    "Faster & More Expensive": (x_vals <= 0) & (y_vals > 0),
    "Faster & Cheaper": (x_vals <= 0) & (y_vals <= 0),
    "Slower & Cheaper": (x_vals > 0) & (y_vals <= 0)
}

quadrant_counts = {k: mask.sum() for k, mask in quadrant_masks.items()}
quadrant_routes = {k: routes.loc[mask, 'route'].tolist() for k, mask in quadrant_masks.items()}

for quadrant, count in quadrant_counts.items():
    print(f"{quadrant}: {count} routes")
    print("Routes:", quadrant_routes[quadrant])
    print()


fig_adjusted = px.scatter(
    routes, x=calculate_time_gap_adjusted(air_overhead=3.0, rail_overhead=0.5), y=price_gap,
    hover_data=['route'],
    title=f"Besides being slower, rail is also more expensive on {quadrant_counts['Slower & More Expensive']} routes",
    labels={'x': 'Time Gap: Rail Time - Air Time (hrs)', 'y': 'Price Gap: Rail Fare - Air Fare (€)'}
)

fig_adjusted.add_vline(x=0, line_dash="dash", line_color="black")
fig_adjusted.add_hline(y=0, line_dash="dash", line_color="black")

# Annotations for each quadrant
x_vals = calculate_time_gap_adjusted(air_overhead=4.0, rail_overhead=0.5)
y_vals = calculate_price_gap_adjusted(baggage_fee=23)

x_min, x_max = x_vals.min(), x_vals.max()
y_min, y_max = y_vals.min(), y_vals.max()

# Midpoints of each quadrant
x_pos_right = (0 + x_max) / 2
x_pos_left  = (0 + x_min-2) / 2
y_pos_top   = (0 + y_max+1) / 2
y_pos_bottom= (0 + y_min-1) / 2

# Add annotations in correct quadrants with route counts
fig_adjusted.add_annotation(
    x=x_pos_right, y=y_pos_top,
    text=f"Slower & <br>More Expensive<br>({quadrant_counts['Slower & More Expensive']} routes)",
    showarrow=False
)
fig_adjusted.add_annotation(
    x=x_pos_left, y=y_pos_top,
    text=f"Faster & <br>More Expensive<br>({quadrant_counts['Faster & More Expensive']} routes)",
    showarrow=False
)
fig_adjusted.add_annotation(
    x=x_pos_left, y=y_pos_bottom,
    text=f"Faster & <br>Cheaper<br>({quadrant_counts['Faster & Cheaper']} routes)",
    showarrow=False
)
fig_adjusted.add_annotation(
    x=x_pos_right, y=y_pos_bottom,
    text=f"Slower & <br>Cheaper<br>({quadrant_counts['Slower & Cheaper']} routes)",
    showarrow=False
)

fig_adjusted.show()

In [0]:
price_per_min_air = routes['air_fare_eur']/routes['air_time_mins']
price_per_min_rail = routes['rail_fare_eur']/routes['rail_time_mins']

In [0]:
# Exploring patterns of total time ans price for rail (this graph) and air (next graph)

fig = px.scatter(
    routes,
    x='rail_time_hrs',
    y='rail_fare_eur',
    hover_data=['route'],
    title='Rail Time vs Rail Price',
    labels={'rail_time_hrs': 'Rail Time (hrs)', 'rail_fare_eur': 'Rail Fare (€)'}
)
fig.show()

In [0]:
# Look for patterns in price per minute

import plotly.express as px

fig = px.scatter(
    routes,
    x='distance_km',
    y=price_per_min_air,
    title='Scatter Plot of Price per Minute (Air) vs Distance',
    labels={'distance_km': 'Distance (km)', 'y': 'Price per Minute (Air) (€)'},
    hover_data=['route']
)
fig.show()

In [0]:
# Look for patterns in price per minute for rail - any obvious differences to "air"?

fig = px.scatter(
    routes,
    x='distance_km',
    y=price_per_min_rail,
    hover_data=['route'],
    title='Scatter Plot of Price per Minute (Rail) vs Distance',
    labels={'distance_km': 'Distance (km)', 'y': 'Price per Minute (Rail) (€)'}
)
fig.show()

In [0]:
routes_play = routes.copy()
routes_play.info()

Generalised cost function

In [0]:
def calculate_generalised_cost(
    routes_play,
    value_of_time=10,
    carbon_price=150,
    airport_overhead=4.0,
    station_overhead=0.35,
    baggage_fee=23,
    emission_value=35, # in grams
    include_carbon=True
):
    df = routes_play.copy()

    # Door-to-door times
    df['rail_total_time'] = df['rail_time_hrs'] + station_overhead
    df['air_total_time'] = df['air_time_hrs'] + airport_overhead

    # Carbon cost
    carbon_factor = carbon_price / 1000 if include_carbon else 0

    # Generalised cost
    df['rail_gc'] = (
        df['rail_fare_eur']
        + df['rail_total_time'] * value_of_time
        + df['distance_km'] * emission_value/1000 * carbon_factor
    )

    df['air_gc'] = (
        df['air_fare_eur']
        + df['cheap_flight'] * baggage_fee
        + df['air_total_time'] * value_of_time
        + df['air_co2_kg'] * carbon_factor
    )

    # Margin
    df['gc_margin'] = df['air_gc'] - df['rail_gc']

    # Winner
    df['winner'] = None
    df.loc[df['gc_margin'] > 0, 'winner'] = 'rail'
    df.loc[df['gc_margin'] <= 0, 'winner'] = 'air'

    return df

Exploration of generalised cost results

In [0]:
# 1 Standard calculation: no carbon, no airport overhead, no station overhead, no baggage fee
# This scenario is the standard calculation when people plan a trip

routes_play_1 = calculate_generalised_cost(routes_play, value_of_time=10, carbon_price=150, airport_overhead=0,station_overhead=0, baggage_fee=0, include_carbon=False)

print(f"Share of routes won by each means of transport: {routes_play_1['winner'].value_counts(normalize=True)}")
print(f"Mean generalised cost margin: {routes_play_1['gc_margin'].mean()}")


In [0]:
# 2 Include airport overhead, station overhead, baggage fee BUT NO CARBON
# This scenario contains 'updated assumptions' about realistic time and money spent on each travel mode

routes_play_2 = calculate_generalised_cost(routes_play, value_of_time=10, carbon_price=150, airport_overhead=3.0,station_overhead=0.5, baggage_fee=23, include_carbon=False)

print(f"Share of routes won by each means of transport: {routes_play_2['winner'].value_counts(normalize=True)}")
print(f"Mean generalised cost margin: {routes_play_2['gc_margin'].mean()}")

In [0]:
# 3 Include airport overhead, station overhead, baggage fee and also carbon emissions

# This is what a complete calculation including the externality would look like: it is not about compensating CO2 for your air or rail travel but instead the "social cost of carbon" (SSC)

routes_play_3 = calculate_generalised_cost(routes_play, value_of_time=10, carbon_price=150, airport_overhead=3.0,station_overhead=0.5, baggage_fee=23, include_carbon=True)

print(f"Share of routes won by each means of transport: {routes_play_3['winner'].value_counts(normalize=True)}")
print(f"Mean generalised cost margin: {routes_play_3['gc_margin'].mean()}")

In [0]:
# Visualise scenario 1 as a bar chart

import plotly.express as px

shares = routes_play_1['winner'].value_counts(normalize=True)
fig = px.bar(
    x=shares.index,
    y=shares.values,
    labels={'x': 'Means of transport', 'y': 'Share'},
    title='Rail vs Air Share (GC1)'
)
fig.update_yaxes(tickformat=".0%")
fig.show()

In [0]:
# Create a bar chart showing the three scenarios at one glance.
# Idea: through sliders, you could have switches for 3 scenarios to directly compare them. Add legend that had the input parameters

# import needed libraries for subplot figures
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def get_shares(df):
    shares = df['winner'].value_counts(normalize=True) # value_counts gets the share as rail-air pair
    return {
        'rail': shares.get('rail', 0),
        'air': shares.get('air', 0)
    }

shares_1 = get_shares(routes_play_1)
shares_2 = get_shares(routes_play_2)
shares_3 = get_shares(routes_play_3)

# Create a figure with subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["GC1", "GC2", "GC3"]
)

fig.add_trace(go.Bar(x=['rail', 'air'], y=[shares_1['rail'], shares_1['air']]), 1, 1)
fig.add_trace(go.Bar(x=['rail', 'air'], y=[shares_2['rail'], shares_2['air']]), 1, 2)
fig.add_trace(go.Bar(x=['rail', 'air'], y=[shares_3['rail'], shares_3['air']]), 1, 3)

fig.update_yaxes(title_text="Share", tickformat=".0%")
fig.update_layout(
    showlegend=False,
    title="Depending on the scenario, the overall winner flips from air to rail",
    height=400,
    width=900
)
fig.show()

In [0]:
# Beauty fixes: subplot titles, colour code the winners

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def get_shares(df):
    shares = df['winner'].value_counts(normalize=True)
    return {
        'rail': shares.get('rail', 0),
        'air': shares.get('air', 0)
    }

# Colour operations thanks to AI helper: function to lighten colours, to be applied on base colours in next step
def lighten_color(hex_color, factor=0.4):
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'#{r:02x}{g:02x}{b:02x}'

base_colors = {'rail': '#1f77b4', 'air': '#9467bd'}
light_colors = {k: lighten_color(v, 0.45) for k, v in base_colors.items()}

shares_1 = get_shares(routes_play_1)
shares_2 = get_shares(routes_play_2)
shares_3 = get_shares(routes_play_3)

fig = make_subplots(rows=1, cols=3, subplot_titles=["Scenario 1", "Scenario 2", "Scenario 3"])

for col, shares in enumerate([shares_1, shares_2, shares_3], start=1): # takes the list and returns pairs of (index, value)
    x = ['rail', 'air']
    y = [shares['rail'], shares['air']]
    colors = [base_colors[label] if value >= 0.5 else light_colors[label]
              for label, value in zip(x, y)]

    fig.add_trace(
        go.Bar(
            x=x,
            y=y,
            marker_color=colors,
            text=[f"{v:.0%}" for v in y],
            textposition="outside",
            cliponaxis=False
        ),
        row=1, col=col
    )

fig.update_yaxes(title_text="Share of routes won", tickformat=".0%")
fig.update_layout(
    showlegend=False,
    title="Depending on the scenario, the overall winner flips from air to rail",
    height=400,
    width=900,
    margin=dict(t=80)
)

fig.show()

In [0]:
# When does rail become preferable? - Scatter plot of distance vs. gc margin
import plotly.express as px
import numpy as np

fig = px.scatter(
    routes_play_3,
    x='distance_km',
    y='gc_margin',
    color='winner',
    hover_data=['route'],
    title='Distance vs Generalised Cost Margin (Scenario 3)',
    labels={'distance_km': 'Distance (km)', 'gc_margin': 'GC Margin (€)', 'winner_co2_kg': 'Winner CO₂ (kg)'}
)
fig.add_hline(y=0, line_width=2, line_color='black')
fig.show()

In [0]:
# add carbon of winner column to adjust scatter size, beauty fix: colours

routes_play_3['winner_co2_kg'] = routes_play_3['air_co2_kg']
routes_play_3.loc[routes_play_3['gc_margin'] > 0, 'winner_co2_kg'] = routes_play_3['rail_co2_kg']

fig = px.scatter(
    routes_play_3,
    x='distance_km',
    y='gc_margin',
    color='winner',
    size='winner_co2_kg',
    hover_data=['route', 'winner_co2_kg'],
    title='Around the threshold of the generalised cost margin, a lot of emissions could be saved',
    labels={'distance_km': 'Distance (km)', 'gc_margin': 'GC Margin (€)', 'winner_co2_kg': 'Winner CO₂ (kg)'},
    color_discrete_map={'rail': '#1f77b4', 'air': '#9467bd'} 
)

fig.update_layout(title_subtitle_text='Distance vs. Generalised Cost Margin in Scenario 3 (including overhead and carbon), <br>Size represents carbon emissions for that trip')
fig.add_hline(y=0, line_width=2, line_color='black')

fig.show()


In [0]:
# Low hanging fruits are the routes where air wins by a slim margin of below 50 euros, so generalised cost margin is between -50 and 0

low_hanging_fruits = routes_play_3[(routes_play_3['gc_margin'] >= -50) & (routes_play_3['gc_margin'] < 0)]

print(low_hanging_fruits.shape)
print(f"These are the routes where air wins by a slim margin of below 50 euros: {low_hanging_fruits['route'].unique}")
print(f"\nThis is the amount of carbon emitted by those routes together (flying each of them once): {low_hanging_fruits['winner_co2_kg'].sum()}")
print(f"\nThis is how much carbon emissions would be saved if the train was taken instead: {low_hanging_fruits['winner_co2_kg'].sum() - low_hanging_fruits['rail_co2_kg'].sum()}")
print(f"\nThis accounts for {round((low_hanging_fruits['winner_co2_kg'].sum() - low_hanging_fruits['rail_co2_kg'].sum()) / low_hanging_fruits['winner_co2_kg'].sum() * 100, 2)}% of the air carbon emissions in this scenario.")

In [0]:
# What is the average generalised cost for the routes on the margin?
low_hanging_fruits.groupby('route').agg({'gc_margin': 'mean'}).describe()

In [0]:
# What is the generalised cost for different distance categories?
import plotly.graph_objects as go

# Calculate mean GC by distance category
gc_means = routes_play_1.groupby('distance_label')[['air_gc', 'rail_gc']].mean().reset_index()

# Create grouped bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    x=gc_means['distance_label'],
    y=gc_means['air_gc'],
    name='Air GC',
    marker_color='red'
))
fig.add_trace(go.Bar(
    x=gc_means['distance_label'],
    y=gc_means['rail_gc'],
    name='Rail GC',
    marker_color='green'
))

fig.update_layout(
    title='Average Generalised Cost by Distance Category (GC1)',
    xaxis_title='Distance Category (1=0-250km, 2=250-500km, 3=500-750km, 4=750+km)',
    yaxis_title='Average Generalised Cost (€)',
    barmode='group'
)
fig.show()

In [0]:
# What is the generalised cost for different distance categories?
# Beauty fixes: colours, title, distance categories as direct labels
import plotly.graph_objects as go

# Calculate mean GC by distance category
gc_means = routes_play_3.groupby('distance_label')[['air_gc', 'rail_gc']].mean().reset_index()

# For nicer labels
distance_map = {1: 'Short', 2: 'Medium', 3: 'Long', 4: 'Ultra Long'}
gc_means['distance_label'] = gc_means['distance_label'].map(distance_map)

# Create grouped bar chart
fig = go.Figure()
fig.add_trace(go.Bar(
    x=gc_means['distance_label'],
    y=gc_means['air_gc'],
    name='Air GC',
    marker_color='#9467bd'
))
fig.add_trace(go.Bar(
    x=gc_means['distance_label'],
    y=gc_means['rail_gc'],
    name='Rail GC',
    marker_color='#1f77b4'
))

fig.update_layout(
    title='Rail is mostly competitive except for ultra long distances',
    xaxis_title='Distance Category (1 = 0-250km, 2 = 250-500km, 3 = 500-750km, 4 = 750+ km)',
    yaxis_title='Average Generalised Cost (€)',
    barmode='group',
    title_subtitle_text='Generalised Cost by Distance Category in Scenario 3'
)

fig.show()

In [0]:
# Histogram of margin
import plotly.express as px

fig = px.histogram(routes_play_1, x='gc_margin', nbins=20,
                   title='Histogram of Generalised Cost Margin (Air - Rail)',
                   labels={'gc_margin': 'GC Margin (€)'})
fig.show()

In [0]:
# Idea: explore histogram of air travel time by category and compare with histogram of rail travel time by category

In [0]:
routes_play.head()

In [0]:
import plotly.express as px

# Sort by gc_margin
routes_play_3_sorted = routes_play_3.sort_values('gc_margin', ascending=True)

fig = px.bar(routes_play_3_sorted,
    x='gc_margin',
    y='route',
    orientation='h',  # horizontal bars
    color='gc_margin',
    title='Generalised cost advantage by route (€)',
    labels={'gc_margin': 'Rail advantage (€)', 'route': ''})

fig.add_vline(x=0, line_width=2, line_color='black')  # zero line
fig.update_layout(showlegend=False, coloraxis_showscale=False)

In [0]:
# Map winner → colors (cleaner than np.where)
routes_play_3_sorted['color'] = routes_play_3_sorted['winner'].map({
    'rail': '#1f77b4',   # Rail wins → Blue
    'air': '#9467bd'     # Air wins → Purple
})

fig = px.bar(routes_play_3_sorted,
    x='gc_margin',
    y='route',
    orientation='h',  # horizontal bars
    color='color',
    color_discrete_map={  # Required for hex codes
        '#1f77b4': '#1f77b4',
        '#9467bd': '#9467bd'
    },
    title='What is the rail advantage of your next route, all things considered?',
    labels={'gc_margin': 'Rail advantage (€)', 'route': ''})

fig.add_vline(x=0, line_width=2, line_color='black')  # zero line
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.update_layout(height=400 + 10 * len(routes_play_3_sorted))
fig.update_yaxes(tickfont=dict(size=10))

Prepare data for streamlit app

In [0]:
# Display the dataframe to download it from the display view - Basis for the streamlit app
display(routes_play)

Predictive Models: Can we find a threshold where rail wins?

In [0]:
# Let's predict the distance threshold where rail wins over air in our scenario 3 model (carbon pricing and 'realistic time evaluation' included)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

# Prepare data: predictor variable and target variable
X = routes_play_3_sorted[['distance_km']]
y = (routes_play_3_sorted['gc_margin'] > 0).astype(int)  # 1 = rail wins, 0 = air wins

# Fit model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = LogisticRegression()
model.fit(X_scaled, y)

# Generate smooth prediction curve
distance_range = np.linspace(routes_play_3_sorted['distance_km'].min(), 
                              routes_play_3_sorted['distance_km'].max(), 300).reshape(-1, 1)
distance_scaled = scaler.transform(distance_range)
prob_rail = model.predict_proba(distance_scaled)[:, 1]

# Find threshold distance where probability = 50%
threshold_idx = np.argmin(np.abs(prob_rail - 0.5))
threshold_distance_s3 = distance_range[threshold_idx][0]
print(f"Rail wins threshold: {threshold_distance_s3:.0f} km")

# Plot
fig = px.scatter(routes_play_3_sorted,
    x='distance_km',
    y='gc_margin',
    color='winner',
    color_discrete_map={'rail': 'green', 'air': 'red'},
    hover_name='route',
    title='Does rail win? The distance threshold',
    labels={'distance_km': 'Distance (km)', 
            'gc_margin': 'Rail advantage (€)'})

# Add probability curve as second y-axis line
fig.add_scatter(x=distance_range.flatten(),
                y=(prob_rail - 0.5) * 200,  # scale to match cost axis roughly
                mode='lines',
                line=dict(color='navy', width=2, dash='dash'),
                name='Rail win probability')

# Add threshold line
fig.add_vline(x=threshold_distance_s3, 
              line_dash='dot', 
              line_color='black',
              annotation_text=f'Threshold: {threshold_distance_s3:.0f}km')

fig.add_hline(y=0, line_width=1, line_color='black')

In [0]:
# Predictive model: Test it for scenario 2

# Prepare data: predictor variable and target variable
X = routes_play_2[['distance_km']]
y = (routes_play_2['gc_margin'] > 0).astype(int)  # 1 = rail wins, 0 = air wins

# Fit model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = LogisticRegression()
model.fit(X_scaled, y)

# Generate smooth prediction curve
distance_range = np.linspace(routes_play_1['distance_km'].min(), 
                              routes_play_1['distance_km'].max(), 300).reshape(-1, 1)
distance_scaled = scaler.transform(distance_range)
prob_rail = model.predict_proba(distance_scaled)[:, 1]

# Find threshold distance where probability = 50%
threshold_idx = np.argmin(np.abs(prob_rail - 0.5))
threshold_distance_s2 = distance_range[threshold_idx][0]
print(f"Rail wins threshold: {threshold_distance_s2:.0f} km")

In [0]:
# Predictive model: Test it for scenario 1

# Prepare data: predictor variable and target variable
X = routes_play_1[['distance_km']]
y = (routes_play_1['gc_margin'] > 0).astype(int)  # 1 = rail wins, 0 = air wins

# Fit model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = LogisticRegression()
model.fit(X_scaled, y)

# Generate smooth prediction curve
distance_range = np.linspace(routes_play_1['distance_km'].min(), 
                              routes_play_1['distance_km'].max(), 300).reshape(-1, 1)
distance_scaled = scaler.transform(distance_range)
prob_rail = model.predict_proba(distance_scaled)[:, 1]

# Find threshold distance where probability = 50%
threshold_idx = np.argmin(np.abs(prob_rail - 0.5))
threshold_distance_s1 = distance_range[threshold_idx][0]
print(f"Rail wins threshold: {threshold_distance_s1:.0f} km")

In [0]:
import plotly.graph_objects as go

# Thresholds found in previous cells
thresholds = [
    threshold_distance_s1,
    threshold_distance_s2,
    threshold_distance_s3
]
scenarios = ['Scenario 1', 'Scenario 2', 'Scenario 3']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=scenarios,
    y=thresholds,
    marker_color=['#1f77b4', '#9467bd', '#2ca02c'],
    text=[f"{t:.0f} km" for t in thresholds],
    textposition='outside'
))

fig.update_layout(
    title='Rail Could Be No. 1 Means of Transport On Middle-distance Routes <br><sup>Distance Threshold Below Which Rail Is On Average More Cost-effective Than Air<sup>',
    xaxis_title='Scenario',
    yaxis_title='Distance Threshold (km)',
    height=500,
    width=600
)
fig.show()

Predictive Models: What is more important, fare or speed?

In [0]:
## Two models: first try is distance only, 'full model' is distance + time gap + price gap
# Use scenario 1 (standard assumptions)

from sklearn.metrics import accuracy_score

# Create binary rail_wins variable from winner column
routes_play_1['rail_wins'] = (routes_play_1['winner'] == 'rail').astype(int)

# Basic: distance only
X1 = routes_play_1[['distance_km']].dropna()
y1 = routes_play_1.loc[X1.index, 'rail_wins']
model1 = LogisticRegression().fit(X1, y1)
print(f"Distance only: {accuracy_score(y1, model1.predict(X1)):.0%}")

# Feature importance for basic model
feature_importance_basic = dict(zip(X1.columns, model1.coef_[0]))
display(feature_importance_basic)

# Expanded: + gaps
X2 = routes_play_1[['distance_km', 'price_gap', 'time_gap']].dropna()
y2 = routes_play_1.loc[X2.index, 'rail_wins']
model2 = LogisticRegression().fit(X2, y2)
print(f"\n\nFull model: {accuracy_score(y2, model2.predict(X2)):.0%}")

# Feature importance for full model
feature_importance_full = dict(zip(X2.columns, model2.coef_[0]))
display(feature_importance_full)
print(f"\n\nHow to read: as distance increases, probability of rail winning increases/decreases by {feature_importance_full['distance_km']:.2f} (distance_km coefficient).\nAs price gap increases (as rail becomes more expnsive than air), probability of rail winning increases/decreases by {feature_importance_full['price_gap']:.2f} (price_gap coefficient).\nAs time gap increases, probability of rail winning increases/decreases by {feature_importance_full['time_gap']:.2f} (time_gap coefficient).")

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2)
model2.fit(X_train, y_train)
print(f"\n\nTest accuracy: {accuracy_score(y_test, model2.predict(X_test)):.0%}")

In [0]:
## Kind of nonsensical because does not include adjusted variables

from sklearn.metrics import accuracy_score

# Create binary rail_wins variable from winner column
routes_play_3['rail_wins'] = (routes_play_3['winner'] == 'rail').astype(int)

# Basic: distance only
X1 = routes_play_3[['distance_km']].dropna()
y1 = routes_play_3.loc[X1.index, 'rail_wins']
model1 = LogisticRegression().fit(X1, y1)
print(f"Distance only: {accuracy_score(y1, model1.predict(X1)):.0%}")

# Feature importance for basic model
feature_importance_basic = dict(zip(X1.columns, model1.coef_[0]))
display(feature_importance_basic)

# Expanded: + gaps
X2 = routes_play_3[['distance_km', 'price_gap', 'time_gap']].dropna()
y2 = routes_play_3.loc[X2.index, 'rail_wins']
model2 = LogisticRegression().fit(X2, y2)
print(f"\n\nFull model: {accuracy_score(y2, model2.predict(X2)):.0%}")

# Feature importance for full model
feature_importance_full = dict(zip(X2.columns, model2.coef_[0]))
display(feature_importance_full)

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2)
model2.fit(X_train, y_train)
print(f"\nTest accuracy: {accuracy_score(y_test, model2.predict(X_test)):.0%}")

In [0]:
## Include adjusted time gap and price gap

# incorporate time gap adjusted and price gap adjusted into the dataframe
routes_play_3_adj = routes_play_3.copy()
routes_play_3_adj['time_gap_adjusted'] = calculate_time_gap_adjusted(air_overhead=3.0, rail_overhead=0.5)
routes_play_3_adj['price_gap_adjusted'] = calculate_price_gap_adjusted(baggage_fee=23)

# Basic: distance only
X1 = routes_play_3[['distance_km']].dropna()
y1 = routes_play_3.loc[X1.index, 'rail_wins']
model1 = LogisticRegression().fit(X1, y1)
print(f"Distance only: {accuracy_score(y1, model1.predict(X1)):.0%}")

# Feature importance for basic model
feature_importance_basic = dict(zip(X1.columns, model1.coef_[0]))
display(feature_importance_basic)

# Expanded: + gaps
X2 = routes_play_3_adj[['distance_km', 'price_gap_adjusted', 'time_gap_adjusted']].dropna()
y2 = routes_play_3_adj.loc[X2.index, 'rail_wins']
model2 = LogisticRegression().fit(X2, y2)
print(f"\n\nFull model: {accuracy_score(y2, model2.predict(X2)):.0%}")

# Feature importance for full model
feature_importance_full = dict(zip(X2.columns, model2.coef_[0]))
display(feature_importance_full)

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2)
model2.fit(X_train, y_train)
print(f"\n\nTest accuracy: {accuracy_score(y_test, model2.predict(X_test)):.0%}")

For better interpretability, standardise variables first

In [0]:
## With standardised units
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

df = routes_play_3_adj.copy()

model_df = df[["distance_km", "price_gap_adjusted", "time_gap_adjusted", "rail_wins"]].dropna()

X = model_df[["distance_km", "price_gap_adjusted", "time_gap_adjusted"]]
y = model_df["rail_wins"]   # replace with your actual binary target column

logit_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression())
])

logit_model.fit(X, y)

coefs = logit_model.named_steps["logit"].coef_[0]

standardised_coefs = pd.Series(coefs, index=X.columns).sort_values(key=abs, ascending=False)

display(standardised_coefs)

What could a decision tree teach us?

In [0]:
## Decision Tree with updated (adjusted) assumptions of scenario 3
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[["distance_km", "price_gap_adjusted", "time_gap_adjusted"]].dropna()
y = (df.loc[X.index, "gc_margin"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

## Feature importance
importances = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

## Decision Tree
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=["air wins", "rail wins"],
    filled=True,
    rounded=True
)
plt.show()

In [0]:
## Decision Tree with non-adjusted inputs time gap and price gap, like in scenario 1

X = routes_play_3[["distance_km", "price_gap", "time_gap"]].dropna()
y = (df.loc[X.index, "rail_wins"] > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

## Feature importance
importances = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

## Decision Tree
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=X.columns,
    class_names=["air wins", "rail wins"],
    filled=True,
    rounded=True
)
plt.show()